# Track B — 취약 행정동 위험등급 레이블 생성 (IMD 방식)

## 개요

본 노트북은 **Track B (분류 모델)** 의 종속변수(레이블)를 생성하는 전처리 파이프라인입니다.

### 핵심 설계 원칙
- **비순환 레이블링**: Track B의 X 피처(독거노인 수, 의료기관 수 등)와 **완전히 독립된** 외부 데이터만 사용
- **영국 IMD(Index of Multiple Deprivation) 방법론 차용**: 다영역 박탈지수로 행정동별 취약도를 객관적으로 측정
- **Andersen 행동모델 기반 가중치**: Need 30% / Enabling×2 25%+25% / Contextual 20%

### 최종 산출물
`df_criteria.csv` — 서울시 420개 행정동 × {고위험 / 중위험 / 저위험} (균형 배분 140개씩)

> **D4 업데이트**: 2021년 기초생계급여 단일 지표 → 2024년 **기초생계급여(94.6%) + 기초의료급여(5.4%)** 통합 지표


## 0. 라이브러리 및 경로 설정

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'
OUT_PATH = 'df_criteria.csv'

## 1. 기준 행정동 목록 구성

`df_analysis_v3.csv`에서 서울시 행정동 코드·명칭·65세 이상 인구를 추출합니다.
이 목록이 레이블 생성의 기준 틀(skeleton)이 됩니다.


In [4]:
df_base = pd.read_csv(f'{DATA_DIR}/df_analysis.csv', encoding='utf-8-sig')
df_base = df_base[['행정동코드', '행정동명', '자치구명', '65세이상인구']].copy()
df_base['행정동코드'] = df_base['행정동코드'].astype(str).str.zfill(10)
df_base = df_base.drop_duplicates(subset='행정동코드')

gu_pop = df_base.groupby('자치구명')['65세이상인구'].sum().reset_index()
gu_pop.columns = ['자치구명', '구_65세이상인구']
seoul_gu = df_base['자치구명'].unique().tolist()

print(f'기준 행정동 수: {len(df_base)}개  |  자치구 수: {df_base["자치구명"].nunique()}개')
df_base.head(5)

기준 행정동 수: 420개  |  자치구 수: 25개


,행정동코드,행정동명,자치구명,65세이상인구
0,1111051500,청운효자동,종로구,2144
1,1111053000,사직동,종로구,1808
2,1111054000,삼청동,종로구,589
3,1111055000,부암동,종로구,1825
4,1111056000,평창동,종로구,3510


## 2. IMD 4개 영역 설계

| 영역 | 데이터 소스 | 지표 | 가중치 | 방향 |
|------|-----------|------|--------|------|
| **D1** 노인 의료 필요도 | 국민건강보험공단 장기요양 등급판정 현황 (2023.12) | 시군구별 65세이상 수급자 비율 | **30%** | ↑ 고위험 |
| **D2** 서비스 접근성 | 국민건강보험공단 장기요양기관 시설별 현황.xlsx | **행정동별** 장기요양기관 부족도 (−기관수/천명, 매핑파일 시) / 자치구 (fallback) | **25%** | ↑ 고위험 |
| **D3** 의료 접근성 | 서울시 지역사회건강통계 질병이환 (2023) | 자치구별 미충족 의료율(%) | **25%** | ↑ 고위험 |
| **D4** 경제적 취약성 | 서울시 기초생활수급자 동별 현황 (2024.05) | 행정동별 65세이상 기초생계+의료급여 수급자 비율 | **20%** | ↑ 고위험 |

> **가중치 근거 (Andersen 행동모델)**  
> - Need: D1(30%) — 실제 의료 필요가 가장 직접적인 취약 지표  
> - Enabling × 2: D2(25%) + D3(25%) — 서비스 공급·접근성이 수요에 영향  
> - Contextual: D4(20%) — 경제적 맥락 (생계+의료급여 통합으로 복합 빈곤 포괄)

> **비순환 설계**: D1~D4 모두 Track B X 피처(독거노인수, 의료기관수, 포화도, 고령화율 등)와 **별개 데이터 소스**


## 3. 영역별 데이터 처리

### D1 — 노인장기요양 등급판정 수급자 비율
**출처**: `2023년 12월 시군구별 등급판정현황(성별, 연령별, 자격별).csv`  
**집계 단위**: 시군구 → 동일 자치구 내 모든 행정동에 동일 값 배정  
**지표**: 65세이상 장기요양 수급자 수 / 65세이상 인구


In [5]:
# 데이터 로드
df1 = pd.read_csv(f'{DATA_DIR}/2023년 12월 시군구별 등급판정현황(성별, 연령별, 자격별).csv', encoding='cp949')

df1_65 = df1[
    (df1['시도'] == '서울') &
    (df1['연령구분'].isin(['65-69세', '70-74세', '75-79세', '80-84세', '85세이상']))
]

grade_cols = ['1등급', '2등급', '3등급', '4등급', '5등급', '인지지원등급']
df1_agg = df1_65.groupby('시군구')[grade_cols].sum()
df1_agg['수급자수'] = df1_agg.sum(axis=1)
df1_agg = df1_agg[['수급자수']].reset_index()
df1_agg.columns = ['자치구명', '수급자수']
df1_agg = df1_agg.merge(gu_pop, on='자치구명')
df1_agg['D1_수급자비율'] = df1_agg['수급자수'] / df1_agg['구_65세이상인구']

print('D1 상위 5개 자치구:')
print(df1_agg.sort_values('D1_수급자비율', ascending=False)[['자치구명', 'D1_수급자비율']].head(5).to_string(index=False))

df_merged = df_base.merge(df1_agg[['자치구명', 'D1_수급자비율']], on='자치구명', how='left')

D1 상위 5개 자치구:
자치구명  D1_수급자비율
 강서구  0.098417
 강동구  0.097405
 구로구  0.092312
 노원구  0.090720
 성북구  0.090538


### D2 — 요양시설 접근성 (장기요양기관 집계)
**출처**: `국민건강보험공단_장기요양기관 시설별 현황.xlsx` (전국 65,412개 / 서울 8,931개)  
**집계 단위**: **행정동** (법정동→행정동 매핑 파일 있을 때) / **자치구** (없을 때 자동 fallback)  
**지표 방향**: 기관 수가 적을수록 서비스 접근성 취약 → 음수 변환으로 고위험 방향 통일  

#### 행정동 레벨 사용 시 — 매핑 파일 1회 다운로드
터미널에서 아래 명령어 실행 후 노트북 재실행:
```bash
curl -o data/code_hdong_bdong.json \
  https://raw.githubusercontent.com/WooilJeong/code/main/code/code_dong/code_hdong_bdong.json
```
- 파일 있음 → 법정동코드 8자리 기준 행정동 매핑 → 행정동별 기관 수 집계  
- 파일 없음 → 시군구코드 기준 자치구 집계 (자동 fallback)


In [ ]:
import json, os
import pandas as pd

NHIS_PATH = f'{DATA_DIR}/국민건강보험공단_장기요양기관 시설별 현황.xlsx'
MAP_PATH  = f'{DATA_DIR}/code_hdong_bdong.json'

# 서울 기관 로드
df_nhis = pd.read_excel(NHIS_PATH, sheet_name='일반현황', header=0, dtype={'장기요양기관코드': str})
df_seoul = df_nhis[df_nhis['시도코드'] == 11].copy()
df_seoul['시군구코드_str'] = df_seoul['시군구코드'].astype(str).str.zfill(3)

# 8자리 법정동코드: 시도(2) + 시군구(3) + 법정동(3)
df_seoul['법정동코드_8'] = '11' + df_seoul['시군구코드_str'] + df_seoul['법정동코드'].astype(str).str.zfill(3)
print(f'서울 장기요양기관: {len(df_seoul)}개')

# 1차: 법정동→행정동 매핑으로 행정동 레벨 집계 
D2_DONG_OK = False
try:
    with open(MAP_PATH, 'r', encoding='utf-8') as f:
        map_raw = json.load(f)
    df_map = pd.DataFrame(map_raw.get('data', []))

    # 유효성 확인 (placeholder / 빈 파일 제외)
    if not {'행정동코드', '법정동코드'}.issubset(df_map.columns) or len(df_map) < 100:
        raise ValueError('매핑 파일 미설치 → data/code_hdong_bdong.json 다운로드 필요')

    df_map_s = df_map[df_map['행정동코드'].astype(str).str.startswith('11')].copy()
    df_map_s['법정동코드_8'] = df_map_s['법정동코드'].astype(str).str[:8]
    df_map_s['행정동코드_10'] = df_map_s['행정동코드'].astype(str).str.zfill(10)

    df_j = df_seoul.merge(
        df_map_s[['법정동코드_8', '행정동코드_10']].drop_duplicates(),
        on='법정동코드_8', how='left'
    )
    matched = df_j['행정동코드_10'].notna().sum()
    print(f'행정동 매핑: {matched}/{len(df_j)} ({matched/len(df_j):.1%})')

    fac_dong = df_j.groupby('행정동코드_10').size().reset_index(name='요양기관수')
    fac_dong.columns = ['행정동코드', '요양기관수']

    df2 = df_base.merge(fac_dong, on='행정동코드', how='left')
    df2['요양기관수'] = df2['요양기관수'].fillna(0)

    # 기관수 0인 행정동 → 자치구 평균 보완
    gu_avg = df2.groupby('자치구명')['요양기관수'].mean().reset_index(name='구_평균')
    df2 = df2.merge(gu_avg, on='자치구명', how='left')
    df2['요양기관수'] = df2['요양기관수'].where(df2['요양기관수'] > 0, df2['구_평균'])
    df2['시설_천명당'] = df2['요양기관수'] / df2['65세이상인구'] * 1000
    df2['D2_시설부족'] = -df2['시설_천명당']

    df_merged = df_merged.merge(df2[['행정동코드', 'D2_시설부족']], on='행정동코드', how='left')
    D2_DONG_OK = True
    print('\n[행정동 ✓] D2 행정동 단위 집계 완료')
    print(df2.sort_values('요양기관수', ascending=False)
          [['행정동명', '자치구명', '요양기관수', 'D2_시설부족']].head(10).to_string(index=False))

except Exception as e:
    print(f'[행정동 불가] {e}')

# 2차: 자치구 레벨 fallback (NHIS 기관 데이터 활용) 
if not D2_DONG_OK:
    print('\n자치구 단위 집계 — NHIS 장기요양기관 8,931개 기준')

    # 시군구코드 → 자치구명 역매핑 (df_base 활용)
    gu_code_map = (df_base.assign(시군구코드=df_base['행정동코드'].str[2:5].astype(int))
                   .drop_duplicates('자치구명')[['자치구명', '시군구코드']]
                   .set_index('시군구코드')['자치구명'].to_dict())
    
    df_seoul['자치구명'] = df_seoul['시군구코드'].map(gu_code_map)

    fac_gu = df_seoul.groupby('자치구명').size().reset_index(name='요양기관수')
    fac_gu = fac_gu.merge(gu_pop, on='자치구명')
    fac_gu['시설_천명당'] = fac_gu['요양기관수'] / fac_gu['구_65세이상인구'] * 1000
    fac_gu['D2_시설부족'] = -fac_gu['시설_천명당']

    print('자치구별 장기요양기관 상위 10:')
    print(fac_gu.sort_values('요양기관수', ascending=False)
          [['자치구명', '요양기관수', '시설_천명당']].head(10).to_string(index=False))

    df_merged = df_merged.merge(fac_gu[['자치구명', 'D2_시설부족']], on='자치구명', how='left')
    print('D2 자치구 단위 집계 완료')


서울 장기요양기관: 8931개
행정동 매핑: 52600/52600 (100.0%)

[행정동 ✓] D2 행정동 단위 집계 완료
  행정동명 자치구명  요양기관수     D2_시설부족
  중계4동  노원구  419.0 -100.915222
  상계9동  노원구  344.0  -92.647455
 상계10동  노원구  344.0 -116.570654
  상계8동  노원구  344.0 -111.724586
상계6.7동  노원구  344.0  -64.856712
  상계5동  노원구  344.0  -65.275142
상계3.4동  노원구  344.0  -55.261044
  상계2동  노원구  344.0  -96.143097
  상계1동  노원구  344.0  -45.049764
  우장산동  강서구  259.0  -41.039455


### D3 — 미충족 의료율
**출처**: `지역사회+건강통계(질병이환)_20260505122351.xlsx` (질병관리청 지역사회건강조사, 2023년)  
**집계 단위**: 자치구 → 행정동 배정  
**지표**: 연간 미충족의료율 조율(%) — 병의원 진료가 필요했으나 받지 못한 경험 비율  

> 파일명의 "질병이환"은 지역사회건강조사 분류 체계상의 상위 카테고리이며,  
> 이 파일에는 미충족의료율 지표가 수록되어 있음 (메타정보 시트 확인).


In [7]:
df3 = pd.read_excel(f'{DATA_DIR}/지역사회+건강통계(질병이환)_20260505122351.xlsx', header=None)

# 파일 구조: row0=공백, row1=컬럼헤더(자치구별/연도), row2=지표명, row3=조율/표준화율, row4+=데이터
# 실제 데이터는 row4부터: col0=자치구(상위), col1=자치구(하위), col2=2023조율, col3=2023표준화율, ...
df3_data = df3.iloc[4:].copy()
df3_data.columns = ['자치구_상위', '자치구명_raw', '미충족의료율_2023_조율', '미충족의료율_2023_표준화',
                    '미충족의료율_2024_조율', '미충족의료율_2024_표준화',
                    '미충족의료율_2025_조율', '미충족의료율_2025_표준화']

# 자치구 합계행(소계) 제거 후 25개 자치구만
df3_data = df3_data[df3_data['자치구명_raw'] != '소계'].copy()
df3_data['자치구명'] = df3_data['자치구명_raw'].astype(str).str.strip()
df3_data['D3_미충족의료율'] = pd.to_numeric(df3_data['미충족의료율_2023_조율'], errors='coerce')
df3_clean = df3_data[df3_data['자치구명'].isin(seoul_gu)][['자치구명', 'D3_미충족의료율']].copy()

print(f'D3 데이터: {len(df3_clean)}개 자치구')
print('\n미충족의료율 상위 5개:')
print(df3_clean.sort_values('D3_미충족의료율', ascending=False).head(5).to_string(index=False))

df_merged = df_merged.merge(df3_clean, on='자치구명', how='left')

D3 데이터: 25개 자치구

미충족의료율 상위 5개:
자치구명  D3_미충족의료율
서대문구       11.5
 노원구        7.0
 금천구        6.1
 동작구        5.4
 성북구        5.2


### D4 — 65세이상 기초수급 비율 (2024년 갱신)
**출처**: `서울시 국민기초생활 수급자 동별 현황(202405).xlsx` (2024-05 기준)  
**집계 단위**: 읍면동 → 행정동명 매칭, 미매칭 시 자치구 평균 보완

#### 급여 구분 활용

| 자격 | 의미 | 65세이상 수급자 | 비중 |
|------|------|----------------|------|
| **기초생계급여** (기본) | 최저생계비 이하 극빈층 | 134,251명 | 94.6% |
| **기초의료급여** (의료) | 의료비 감당 불가 저소득층 | 7,658명 | 5.4% |
| 기초주거급여, 기초교육급여 | 미사용 (목적 상이) | — | — |

> 두 급여 모두 경제적 취약성에서 기인하며, 합산 시 노인 빈곤의 생계·의료 양면을 포괄하는 복합 지표가 됨.

#### 행정동명 매칭 전략
1. **직접 매칭** (241개): 읍면동명 = 행정동명
2. **규칙 변환** (177개): 숫자 앞 '제' 삽입 (`창신1동` → `창신제1동`)
3. **수동 매핑** (9개): 특수 패턴 (`금호2-3가동`→`금호2.3가동`, `홍제1동`→`홍제제1동` 등)
4. **자치구 평균 fallback** (6개): `강일동`, `개포3동`, `둔촌1동`, `상일제1·2동`, `항동` (행정구역 개편으로 df_base 미존재)


In [8]:
df_raw = pd.read_excel(f'{DATA_DIR}/서울시 국민기초생활 수급자 동별 현황(202405).xlsx', header=None)

df_raw.columns = ['읍면동', '자격', '연령구간', 'col3', '수급권자수']
df_raw = df_raw.iloc[2:].copy()
df_raw['읍면동'] = df_raw['읍면동'].ffill()
df_raw['자격'] = df_raw['자격'].ffill()
df_raw = df_raw[df_raw['연령구간'].notna()].copy()
df_raw['수급권자수'] = pd.to_numeric(df_raw['수급권자수'], errors='coerce').fillna(0).astype(int)

# 자치구 합계 행 제거 + 65세이상 생계·의료급여 필터
df_dong = df_raw[~df_raw['읍면동'].isin(seoul_gu)].copy()
df_dong['읍면동'] = df_dong['읍면동'].str.strip()
df_65 = df_dong[
    (df_dong['연령구간'] == '65세이상') &
    (df_dong['자격'].isin(['기초생계급여', '기초의료급여']))
].copy()

# 급여 구성 비율 출력
breakdown = df_65.groupby('자격')['수급권자수'].sum()
for z, cnt in breakdown.items():
    print(f'{z}: {cnt:,}명 ({cnt/breakdown.sum()*100:.1f}%)')

# 읍면동별 합산
dong_combined = df_65.groupby('읍면동')['수급권자수'].sum().reset_index()
dong_combined.columns = ['읍면동_raw', '노인수급자수_합산']

# 행정동명 정규화 매핑
base_dongs = set(df_base['행정동명'].str.strip().unique())

MANUAL_MAP = {
    '금호2-3가동': '금호2.3가동',   '성수1가1동': '성수1가제1동',
    '성수1가2동':  '성수1가제2동',  '성수2가1동': '성수2가제1동',
    '성수2가3동':  '성수2가제3동',  '중계2,3동':  '중계2.3동',
    '홍제1동':    '홍제제1동',      '홍제2동':   '홍제제2동',
    '홍제3동':    '홍제제3동',
    # '항동'은 구로구 소속이나 df_base 미존재 → fallback
}

def normalize_dong(name):
    name = str(name).strip()
    if name in MANUAL_MAP:   
        return MANUAL_MAP[name]
    if name in base_dongs:   
        return name
    norm = re.sub(r'([가-힣])(\d)', r'\1제\2', name)
    return norm if norm in base_dongs else None

dong_combined['행정동명'] = dong_combined['읍면동_raw'].apply(normalize_dong)
matched = dong_combined['행정동명'].notna().sum()
fallback = dong_combined['행정동명'].isna().sum()
print(f'\n행정동 매칭: {matched}개 직접 / {fallback}개 자치구평균 fallback')

# df_base 조인 후 미매칭 → 자치구 평균 보완
df4_joined = df_base.merge(dong_combined[['행정동명', '노인수급자수_합산']], on='행정동명', how='left')
gu_total = df4_joined.groupby('자치구명')['노인수급자수_합산'].sum().reset_index(name='구_수급자합')
gu_total = gu_total.merge(gu_pop, on='자치구명')
gu_total['구_수급자비율'] = gu_total['구_수급자합'] / gu_total['구_65세이상인구']
df4_joined = df4_joined.merge(gu_total[['자치구명', '구_수급자비율']], on='자치구명', how='left')
df4_joined['D4_수급비율(2024)'] = (
    df4_joined['노인수급자수_합산'] / df4_joined['65세이상인구']
).fillna(df4_joined['구_수급자비율'])

df_merged = df_merged.merge(df4_joined[['행정동코드', 'D4_수급비율(2024)']], on='행정동코드', how='left')
print(f'최종 결측: {df_merged["D4_수급비율(2024)"].isna().sum()}개')

기초생계급여: 134,251명 (94.6%)
기초의료급여: 7,658명 (5.4%)

행정동 매칭: 418개 직접 / 6개 자치구평균 fallback
최종 결측: 0개


### 중간 검증 — 4개 도메인 결측 및 기술통계

In [9]:
domain_cols = ['D1_수급자비율', 'D2_시설부족', 'D3_미충족의료율', 'D4_수급비율(2024)']

print('결측치 확인:')
print(df_merged[domain_cols].isnull().sum().to_string())

결측치 확인:
D1_수급자비율         0
D2_시설부족          0
D3_미충족의료율        0
D4_수급비율(2024)    0


## 4. IMD 종합점수 산출

### 방법론 (영국 DCLG IMD 2015 준용)
1. **Z-score 정규화**: 각 영역 지표를 평균 0, 표준편차 1로 변환
2. **지수 변환 (exp)**: 양수화 및 분포 왜도 보정 — 순위 보존 + 영역 간 상쇄 방지
3. **가중합**: 30% × D1 + 25% × D2 + 25% × D3 + 20% × D4
4. **3분위 분류**: 상위 1/3 → 고위험 / 중간 → 중위험 / 하위 1/3 → 저위험


In [10]:
WEIGHTS = {
    'D1_수급자비율':     0.30,
    'D2_시설부족':       0.25,   # 이미 음수 변환 완료
    'D3_미충족의료율':   0.25,
    'D4_수급비율(2024)': 0.20,
}

for col in WEIGHTS:
    mu, sigma = df_merged[col].mean(), df_merged[col].std()
    df_merged[f'{col}_z']   = (df_merged[col] - mu) / sigma
    df_merged[f'{col}_exp'] = np.exp(df_merged[f'{col}_z'])
    print(f'{col}: μ={mu:.4f}, σ={sigma:.4f}')

df_merged['IMD_score'] = sum(
    w * df_merged[f'{col}_exp'] for col, w in WEIGHTS.items()
)

print()
print('IMD 종합점수 기술통계:')
print(df_merged['IMD_score'].describe().round(4))

D1_수급자비율: μ=0.0821, σ=0.0094
D2_시설부족: μ=-21.3125, σ=24.4798
D3_미충족의료율: μ=4.0688, σ=1.9119
D4_수급비율(2024): μ=0.0819, σ=0.0551

IMD 종합점수 기술통계:
count    420.0000
mean       2.1963
std        4.8537
min        0.2235
25%        0.8957
50%        1.1579
75%        1.7872
max       60.9581
Name: IMD_score, dtype: float64


## 5. 3분위 분류 → 위험등급 레이블 생성

3분위(tertile) 방식 적용 → 각 등급 정확히 140개 (균형 레이블)


In [11]:
q33 = df_merged['IMD_score'].quantile(1/3)
q67 = df_merged['IMD_score'].quantile(2/3)
print(f'1/3 분위수 (저/중 경계): {q33:.4f}')
print(f'2/3 분위수 (중/고 경계): {q67:.4f}')

df_merged['위험등급'] = pd.cut(
    df_merged['IMD_score'],
    bins=[-np.inf, q33, q67, np.inf],
    labels=['저위험', '중위험', '고위험']
)

print('\n위험등급 분포:')
print(df_merged['위험등급'].value_counts().sort_index())

1/3 분위수 (저/중 경계): 0.9588
2/3 분위수 (중/고 경계): 1.6077

위험등급 분포:
위험등급
저위험    140
중위험    140
고위험    140
Name: count, dtype: int64


## 6. 최종 데이터 저장

In [12]:
output_cols = [
    '행정동코드', '행정동명', '자치구명',
    'D1_수급자비율', 'D2_시설부족', 'D3_미충족의료율', 'D4_수급비율(2024)',
    'D1_수급자비율_z', 'D2_시설부족_z', 'D3_미충족의료율_z', 'D4_수급비율(2024)_z',
    'IMD_score', '위험등급'
]
df_output = df_merged[output_cols].sort_values(['자치구명', '행정동명']).reset_index(drop=True)
# df_output.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print(f'저장 완료: {OUT_PATH}')
print(f'총 {len(df_output)}개 행정동  |  고유 코드: {df_output["행정동코드"].nunique()}개')
df_output.head(10)

저장 완료: df_criteria.csv
총 420개 행정동  |  고유 코드: 420개


,행정동코드,행정동명,자치구명,D1_수급자비율,D2_시설부족,D3_미충족의료율,D4_수급비율(2024),D1_수급자비율_z,D2_시설부족_z,D3_미충족의료율_z,D4_수급비율(2024)_z,IMD_score,위험등급
0,1168066000,개포1동,강남구,0.070774,-41.095890,3.1,0.041691,-1.210967,-0.808149,-0.506729,-0.729876,0.447803,저위험
1,1168067000,개포2동,강남구,0.070774,-27.188825,3.1,0.018708,-1.210967,-0.240047,-0.506729,-1.147199,0.500141,저위험
2,1168069000,개포4동,강남구,0.070774,-19.519095,3.1,0.060820,-1.210967,0.073261,-0.506729,-0.382544,0.645416,저위험
3,1168052100,논현1동,강남구,0.070774,-12.329224,3.1,0.075308,-1.210967,0.366967,-0.506729,-0.119481,0.778302,저위험
4,1168053100,논현2동,강남구,0.070774,-11.562500,3.1,0.035000,-1.210967,0.398288,-0.506729,-0.851376,0.697672,저위험
5,1168060000,대치1동,강남구,0.070774,-10.705057,3.1,0.001107,-1.210967,0.433314,-0.506729,-1.466779,0.671712,저위험
6,1168061000,대치2동,강남구,0.070774,-3.710575,3.1,0.006122,-1.210967,0.719038,-0.506729,-1.375719,0.803635,저위험
7,1168063000,대치4동,강남구,0.070774,-9.532888,3.1,0.054814,-1.210967,0.481197,-0.506729,-0.491602,0.766820,저위험
8,1168065500,도곡1동,강남구,0.070774,-2.923027,3.1,0.009743,-1.210967,0.751210,-0.506729,-1.309971,0.823845,저위험
9,1168065600,도곡2동,강남구,0.070774,-1.743510,3.1,0.004068,-1.210967,0.799393,-0.506729,-1.413019,0.844717,저위험


## 7. 결과 검증

In [13]:
# 7-1. 자치구별 위험등급 분포
summary = df_output.groupby(['자치구명', '위험등급']).size().unstack(fill_value=0)
for col in ['저위험', '중위험', '고위험']:
    if col not in summary.columns: summary[col] = 0
summary = summary[['저위험', '중위험', '고위험']]
summary['합계'] = summary.sum(axis=1)
summary['고위험비율'] = (summary['고위험'] / summary['합계']).round(2)
print('자치구별 위험등급 분포:')
print(summary.to_string())

자치구별 위험등급 분포:
위험등급  저위험  중위험  고위험  합계  고위험비율
자치구명                          
강남구    17    2    2  21   0.10
강동구     0    0   15  15   1.00
강북구     1   10    2  13   0.15
강서구     0    0   20  20   1.00
관악구    14    7    0  21   0.00
광진구     7    8    0  15   0.00
구로구     0    5   10  15   0.67
금천구     0    0   10  10   1.00
노원구     0    0   19  19   1.00
도봉구     0   13    1  14   0.07
동대문구    2   11    1  14   0.07
동작구     3   12    0  15   0.00
마포구    13    3    0  16   0.00
서대문구    0    0   14  14   1.00
서초구    18    0    0  18   0.00
성동구    12    5    0  17   0.00
성북구     0    1   19  20   0.95
송파구    18    9    0  27   0.00
양천구    14    4    0  18   0.00
영등포구    1   16    1  18   0.06
용산구     9    6    1  16   0.06
은평구     0    7    9  16   0.56
종로구     2   13    2  17   0.12
중구      9    5    1  15   0.07
중랑구     0    3   13  16   0.81


In [14]:
# 7-2. 고위험 / 저위험 대표 행정동
print('고위험 상위 10개')
print(df_output[df_output['위험등급'] == '고위험']
      .sort_values('IMD_score', ascending=False)
      [['행정동명', '자치구명', 'D4_수급비율(2024)', 'IMD_score']]
      .head(10).to_string(index=False))

print('\n저위험 상위 10개')
print(df_output[df_output['위험등급'] == '저위험']
      .sort_values('IMD_score')
      [['행정동명', '자치구명', 'D4_수급비율(2024)', 'IMD_score']]
      .head(10).to_string(index=False))

고위험 상위 10개
  행정동명 자치구명  D4_수급비율(2024)  IMD_score
 가양제2동  강서구       0.394832  60.958145
 등촌제3동  강서구       0.373384  41.901650
   수서동  강남구       0.370475  38.459773
   남영동  용산구       0.369270  37.579952
   천연동 서대문구       0.107514  13.354632
남가좌제2동 서대문구       0.098679  13.259185
북가좌제2동 서대문구       0.083486  13.149429
  북아현동 서대문구       0.078440  13.148148
   충현동 서대문구       0.081609  13.108672
   신촌동 서대문구       0.046248  13.096792

저위험 상위 10개
행정동명 자치구명  D4_수급비율(2024)  IMD_score
반포본동  서초구       0.037721   0.223469
반포4동  서초구       0.011885   0.382217
서초3동  서초구       0.020986   0.393976
개포1동  강남구       0.041691   0.447803
서초2동  서초구       0.013603   0.449572
방배3동  서초구       0.013835   0.450859
방배본동  서초구       0.031008   0.465506
방배1동  서초구       0.041005   0.467269
서초1동  서초구       0.043011   0.471118
반포1동  서초구       0.034219   0.480630


In [15]:
# 7-3. 도메인 간 상관계수 (독립성 검증)
domain_cols = ['D1_수급자비율', 'D2_시설부족', 'D3_미충족의료율', 'D4_수급비율(2024)']
print('도메인 간 상관계수:')
print(df_output[domain_cols].corr().round(3).to_string())
print()
corr = df_output[domain_cols].corr().abs()
max_offdiag = corr.where(corr < 1).stack().max()
print(f'\n최대 도메인간 상관계수(절대값): {max_offdiag:.3f}')
if max_offdiag < 0.6:
    print('모든 도메인 쌍 절대값 0.6 미만 — 독립적 정보 제공')
elif max_offdiag < 0.8:
    print('일부 도메인 상관계수 0.6 이상 — 부분 중복 존재 (IMD 방법론상 허용 범위)')
else:
    print('도메인간 강한 상관 — D2 레벨 업그레이드 또는 지표 재검토 필요')

도메인 간 상관계수:
               D1_수급자비율  D2_시설부족  D3_미충족의료율  D4_수급비율(2024)
D1_수급자비율          1.000   -0.152      0.180          0.238
D2_시설부족          -0.152    1.000     -0.046         -0.009
D3_미충족의료율         0.180   -0.046      1.000          0.062
D4_수급비율(2024)     0.238   -0.009      0.062          1.000


최대 도메인간 상관계수(절대값): 0.238
모든 도메인 쌍 절대값 0.6 미만 — 독립적 정보 제공


## 8. 방법론 한계 및 주의사항

1. **공간 해상도 불일치**: D1·D3은 **자치구 단위**, D2는 **행정동 단위(API)** → D4(행정동 단위)와 함께 구내 변이를 일부 포착  
   → D1·D3의 자치구 평균 배정 한계는 Track B 학습 후 피처 중요도로 보완 확인 권장

2. **D2 법정동→행정동 매핑 한계**: 법정동과 행정동은 1:1 대응이 아님. `code_hdong_bdong.json` 미설치 시 자치구 평균 배정으로 전환됨

3. **D4 급여 중복 가능성**: 기초생계급여와 기초의료급여 수급자가 동일 인물일 수 있음 (서울 전체 약 5.4% 규모)  
   → 두 급여의 선정 기준이 다르므로 실질 중복 제한적; 합산 시 소폭 과추정 가능성 존재

4. **행정구역 미매칭 6개**: `강일동`, `개포3동`, `둔촌1동`, `상일제1·2동`, `항동`  
   → df_analysis_v3 기준 행정동 목록에 없는 신설·폐지 행정구역, 자치구 평균 적용

5. **레이블 독립성**: D1~D4 모두 Track B X 피처와 **별개 데이터 소스** → 순환 레이블링 없음  
   단, 기저 구조(고령화율 ↔ D1 잠재 상관) 가능성 — 모델 학습 후 피처 중요도로 확인 권장
